In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.lines as mlines
import matplotlib.ticker as mticker
import matplotlib.lines as mlines

In [ ]:
results_path = '/home/jimlaudagostini/amrex_LB/output/'
result_files = []

for result in os.walk(results_path):
    if 'laser-ion-' in result[0]:
        for f in result[2]:
            if '.csv' in f and not 'Plasma' in f and not 'dist' in f:
                file = os.path.join(result[0], f)
                result_files.append(file)

def custom_sort(file_names):
    def parse_file_name(file_name):
        base_name = os.path.basename(file_name)
        match = re.match(r'laser-ion-(\d+)-(\d+)(.*)', base_name)
        if match:
            return (int(match.group(1)), int(match.group(2)))
        else:
            raise ValueError(f"Invalid file name format: {file_name}")

    return sorted(file_names, key=parse_file_name)

result_files = custom_sort(result_files)
result_files

In [ ]:
to_concat = []

for result in result_files:
    df = pd.read_csv(result, header=None)
    source = re.sub(".csv", "", os.path.basename(result))
    df['Source'] = re.sub("laser-ion-([0-9]*)-1740", r"\1", source)
    to_concat.append(df)

combined_df = pd.concat(to_concat, ignore_index=True, axis=0)
combined_df

In [ ]:
combined_df.columns = ['Run', 'Algorithm', 'Metric', 'Value', 'Rank', 'Source']
combined_df['Metric'] = combined_df['Metric'].str.strip()
combined_df['Algorithm'] = combined_df['Algorithm'].str.strip()
df_eff = combined_df[combined_df['Metric'].isin(['Efficiency','PreviousEfficiency'])]
df_eff

In [ ]:
df_done = df_eff.pivot_table(index=['Run', 'Algorithm', 'Source'],
                          columns='Metric',
                          values='Value').reset_index()
df_done.columns.name = None
df_done = df_done.reset_index(drop=True)
df_done.fillna(0, inplace=True)
df_done['Source'] = pd.Categorical(df_done['Source'], categories=['8', '16', '32', '48', '96'], ordered=True)
df_done['Algorithm'] = pd.Categorical(df_done['Algorithm'], categories=['Original', 'Knapsack', 'SFC', 'Karmarkar-Karp', 'SFC+Painter', 'SFC+Knapsack', 'Painter+Knapsack'], ordered=True)
df_done

In [ ]:
# Plot the average efficiency
plt.figure(figsize=(8, 5))
sns.boxplot(x='Source', y='Efficiency', hue='Algorithm', data=df_done, palette='Set1')

# Add a grid with a multiple of 1
plt.grid(True, which='both', axis='both', linestyle='--', color='gray')

# Customize the plot
plt.title('Efficiency of Algorithms')
plt.xlabel('Total MPI Ranks')
plt.ylabel('Efficiency')
# plt.xticks(rotation=35)  # Rotate x-axis labels for better readability
plt.legend(title='Algorithm', loc='lower left')  # Place legend outside the plot
plt.ylim(0.7, 1.05)

# Show the plot
plt.tight_layout()

plt.savefig('average_efficiency.png', dpi=300)
plt.show()

In [ ]:
weights = combined_df[combined_df['Metric'] == 'Weight']
weights

In [ ]:
df_weights = weights.pivot_table(index=['Run', 'Algorithm', 'Source', 'Rank'],
                          columns='Metric',
                          values='Value').reset_index()
df_weights.columns.name = None
df_weights.reset_index(drop=True, inplace=True)
df_weights.fillna(0, inplace=True)
df_weights['Source'] = pd.Categorical(df_weights['Source'], categories=['8', '16', '32', '48', '96'], ordered=True)
df_weights['Algorithm'] = pd.Categorical(df_weights['Algorithm'], categories=['Original', 'Knapsack', 'SFC', 'Karmarkar-Karp', 'SFC+Painter', 'SFC+Knapsack', 'Painter+Knapsack'], ordered=True)
df_weights

In [ ]:
df_weights[(df_weights['Algorithm'] == 'Karmarkar-Karp') & (df_weights['Source'] == '8') & (df_weights['Run'] == 0)]

In [ ]:
df_melted = df_done.melt(id_vars=['Run', 'Algorithm', 'Source'], value_vars=['Efficiency', 'PreviousEfficiency'],
                    var_name='Metric', value_name='Value')

df_melted['Source'] = pd.Categorical(df_melted['Source'], categories=custom_sort(df_melted['Source'].unique()), ordered=True)
df_done['Algorithm'] = pd.Categorical(df_done['Algorithm'], categories=['Original', 'Knapsack', 'SFC', 'Karmarkar-Karp', 'SFC+Painter', 'SFC+Knapsack', 'Painter+Knapsack'], ordered=True)
df_melted

In [ ]:
df_weights.loc[:,'Log_Weight'] = np.log(df_weights['Weight'])
df_weights['Mean_Weight'] = df_weights.groupby(['Run', 'Algorithm', 'Source'])['Log_Weight'].transform('mean')
df_weights['Diff'] = df_weights['Log_Weight'] - df_weights['Mean_Weight']
df_weights

In [ ]:
sources = df_weights[df_weights['Source'].isin(['8', '96'])]['Source'].unique()
algorithms = df_weights[df_weights['Algorithm'] != 'Original']['Algorithm'].unique()

In [ ]:
# Create a figure with subplots
rows = int(len(algorithms)/3)
cols = int(len(algorithms)/2)
fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(4*cols, 4*rows))
algo = 0
tick_locations = [int(x) for x in np.linspace(min(df_weights['Rank']), max(df_weights['Rank']), 10)]
# Iterate over sources and create a subplot for each
for a in range(rows):
    for i in range(cols):
        # print(algorithms[algo])
        source_df = df_weights[(df_weights['Source'] == '96') & (df_weights['Algorithm'] == algorithms[algo]) & (df_weights['Run'] == 50)]
        sns.scatterplot(data=source_df, x="Rank", y="Diff", ax=axes[a][i])
        axes[a][i].axhline(0, color='black')
        axes[a][i].set_xticks(tick_locations)
        axes[a][i].set_xticklabels(tick_locations)
        axes[a][i].tick_params(axis='x', labelrotation=45)
        axes[a][i].set_title(algorithms[algo])
        axes[a][i].set_xlabel('Rank')
        axes[a][i].set_ylabel('Distance (log scale)')
        algo += 1
# Layout so plots do not overlap
fig.tight_layout(rect=[0, 0.03, 1, 0.95])  # make space for the title
fig.suptitle('Distance From Ideal Load (Step 5000)')
# Show the plot
plt.show()
fig.savefig('distance_from_ideal.png', dpi=300)

In [ ]:
algorithms

In [ ]:
original = df_done[df_done['Algorithm'] == 'Original']
original

In [ ]:
original_melt = original.melt(id_vars=['Run', 'Source'], value_vars=['PreviousEfficiency'],
                    var_name='Metric', value_name='Value')
original_melt['Metric'] = 'Original'
original_melt

In [ ]:
merged_df = original_melt.merge(df_melted, on=['Run', 'Source'], how='inner')
merged_df

In [ ]:
plt.figure(figsize=(8, 4))
# g = sns.FacetGrid(df_melted, col='Algorithm', col_wrap=3, height=5, aspect=1.2, sharey=True)
# g.map(sns.lineplot, 'Run', 'Value', 'Metric', markers=True, dashes=False, palette={'Efficiency': 'blue', 'PreviousEfficiency': 'orange'})
sns.lineplot(x='Run', y='Value', hue='Algorithm', data=df_melted, errorbar=None, markers=True, dashes=False)
# Show the plot
plt.show()

Let's get the weight distribution across different executions

In [ ]:
inputs_path = '/home/jimlaudagostini/amrex_LB/input/'
input_files = []

for result in os.walk(inputs_path):
    if 'laser-ion-' in result[0]:
        for f in result[2]:
            if '.csv' in f and 'original' not in f:
                file = os.path.join(result[0], f)
                input_files.append(file)

def custom_sort(file_names):
    def parse_file_name(file_name):
        base_name = os.path.basename(file_name)
        match = re.match(r'laser-ion-(\d+)-(\d+)(.*)', base_name)
        if match:
            return (int(match.group(1)), int(match.group(2)))
        else:
            raise ValueError(f"Invalid file name format: {file_name}")

    return sorted(file_names, key=parse_file_name)

input_files = custom_sort(input_files)
input_files

In [ ]:
to_concat = []

for result in input_files:
    df = pd.read_csv(input_files[0], header=None)
    df = df.iloc[::2]
    df = df.T.reset_index()
    df.columns = ['Box ID'] + [f'Step {i}' for i in range(51)]
    df = df.stack().reset_index()
    df.columns = ['Box ID', 'Step', 'Weight']
    
    df = df[df['Step'] != 'Box ID']
    df['Step'] = df['Step'].str.replace('Step ', '').astype(int)
    df['Step'] = df['Step'] * 100
    source = re.sub(".csv", "", os.path.basename(result))
    df['Source'] = re.sub("laser-ion-([0-9]*)-1740", r"\1", source)
    to_concat.append(df)

inputs_df = pd.concat(to_concat, ignore_index=True, axis=0)
inputs_df

In [ ]:
inputs_df.loc[:,'Log_Weight'] = np.log(inputs_df['Weight'])
df_count = inputs_df.groupby(['Source', 'Step', 'Log_Weight']).size().reset_index(name='Count')
df_8 = df_count[(df_count['Source'] == '8')]
filtered_df = df_8[df_8['Step'].isin([0, 100, 2500, 5000])]
filtered_df

In [ ]:
plt.figure(figsize=(8, 5))
sns.set_style("whitegrid")
colors = ['red', 'green', 'blue', 'orange', 'pink']
linestyles = ['-', '--', '-.', ':', (0, (3, 1, 1, 1, 1, 1))]

for i, step in enumerate(filtered_df['Step'].unique()):
    source_df = filtered_df[filtered_df['Step'] == step]
    plt.plot(source_df['Log_Weight'], source_df['Count'], 
             label=f"Step {step}", 
             color=colors[i % len(colors)], 
             linestyle=linestyles[i % len(linestyles)])

plt.xlabel('Box Weight (in log scale)')
plt.ylabel('Frequency')
plt.title('Frequency of box weights in different steps')
plt.legend()
# tick_locations = np.linspace(min(filtered_df['Log_Weight']), max(filtered_df['Log_Weight']), 20)  # Show 10 ticks
# plt.xticks(tick_locations, rotation=45)
plt.tight_layout()
plt.savefig('freqeuncy_of_weights.png', dpi=300)
plt.show()

In [ ]:
g = sns.FacetGrid(filtered_df, col="Step")
g.map(plt.plot, "Weight", "Count")

for ax in g.axes.flatten():
    ax.set_xticks(np.linspace(min(filtered_df['Weight']), max(filtered_df['Weight']), 20))
    ax.set_xticklabels(np.linspace(min(filtered_df['Weight']), max(filtered_df['Weight']), 20), rotation=45)

plt.tight_layout()
plt.show()